# Notebook 09: Manuscript Figures and Tables
Opx ML Thermobarometer
Author: [Your name]
Date: 2026-04-04

This notebook compiles all publication-quality figures and tables for the manuscript.

In [1]:
%matplotlib inline
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
from config import (ROOT, DATA_RAW, DATA_PROC, DATA_SPLITS, DATA_NATURAL,
                    MODELS, FIGURES, RESULTS, LOGS, EXPETDB)

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica'],
    'font.size': 10,
    'axes.labelsize': 10,
    'axes.titlesize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.titlesize': 12
})

## 9.1 Table 1: Dataset Summary

In [2]:
df_core = pd.read_csv(DATA_PROC / 'opx_clean_core.csv')
df_full = pd.read_csv(DATA_PROC / 'opx_clean_full.csv')

table1 = pd.DataFrame({
    'Dataset': ['Core (5 oxides)', 'Full (9 oxides)'],
    'n_samples': [len(df_core), len(df_full)],
    'n_experiments': [df_core['Experiment'].nunique(), df_full['Experiment'].nunique()],
    'n_studies': [df_core['Citation'].nunique(), df_full['Citation'].nunique()],
    'T_range_C': [f"{df_core['T_C'].min():.0f}-{df_core['T_C'].max():.0f}",
                  f"{df_full['T_C'].min():.0f}-{df_full['T_C'].max():.0f}"],
    'P_range_kbar': [f"{df_core['P_kbar'].min():.1f}-{df_core['P_kbar'].max():.1f}",
                     f"{df_full['P_kbar'].min():.1f}-{df_full['P_kbar'].max():.1f}"],
    'Mg_range': [f"{df_core['Mg_num'].min():.2f}-{df_core['Mg_num'].max():.2f}",
                 f"{df_full['Mg_num'].min():.2f}-{df_full['Mg_num'].max():.2f}"]
})

table1.to_csv(RESULTS / 'table1_dataset_summary.csv', index=False)
print('Table 1: Dataset Summary')
print(table1)

Table 1: Dataset Summary
           Dataset  n_samples  n_experiments  n_studies T_range_C  \
0  Core (5 oxides)       1148           1034        126  550-1850   
1  Full (9 oxides)        525            493         68  599-1850   

  P_range_kbar   Mg_range  
0    0.0-100.0  0.41-1.00  
1    0.0-100.0  0.56-1.00  


## 9.2 Table 2: Model Performance (from nb03)

In [3]:
results_df = pd.read_csv(RESULTS / 'nb03_results_summary.csv')

table2 = results_df.pivot_table(
    values=['rmse_test', 'mae_test', 'r2_test'],
    index=['target', 'model_type'],
    columns='model',
    aggfunc='first'
)

table2.to_csv(RESULTS / 'table2_model_performance.csv')
print('Table 2: Model Performance')
print(table2.round(3))

Table 2: Model Performance
                  mae_test                            r2_test                \
model                  ERT       GB       RF      XGB     ERT     GB     RF   
target model_type                                                             
P_kbar opx_liq       4.311    3.877    4.281    4.422   0.558  0.689  0.621   
       opx_only     10.406   10.676   10.213   10.163   0.139  0.123  0.168   
T_C    opx_liq      69.100   70.060   67.925   68.442   0.427  0.439  0.442   
       opx_only    142.460  141.342  148.385  149.521   0.259  0.273  0.202   

                         rmse_test                             
model                XGB       ERT       GB       RF      XGB  
target model_type                                              
P_kbar opx_liq     0.585     6.636    5.566    6.145    6.427  
       opx_only    0.163    15.703   15.851   15.444   15.486  
T_C    opx_liq     0.447   125.711  124.399  124.018  123.512  
       opx_only    0.197   186.725 

## 9.3 Table 3: Putirka Comparison (from nb04)

In [4]:
comparison_df = pd.read_csv(RESULTS / 'nb04_putirka_comparison.csv')
print('Table 3: Putirka vs ML Comparison')
print(comparison_df.round(3))

Table 3: Putirka vs ML Comparison
                           Method Target     RMSE      MAE     R2  n_test
0            Putirka 28a (true P)      T  331.683  311.443 -2.989     170
1            Putirka 28b (true P)      T  320.058  295.324 -2.714     170
2            Putirka 29a (true T)      P    7.937    5.884  0.359     143
3  Putirka 29c (opx-only, true T)      P   19.023    9.940 -4.267     144
4     Putirka iterative (28a+29a)      T  120.866   61.141  0.472     143
5     Putirka iterative (28a+29a)      P    5.402    3.622  0.703     143
6              Putirka 28a (ML P)      T  328.600  307.904 -2.915     170
7              Putirka 29a (ML T)      P    8.023    5.857  0.345     143
8                   ML-RF opx-liq      T  124.018   67.925  0.442     170
9                   ML-RF opx-liq      P    6.145    4.281  0.621     170


## 9.4 Table 4: LOSO Results (from nb05)

In [5]:
loso_df = pd.read_csv(RESULTS / 'nb05_loso_all.csv')

table4 = loso_df.groupby(['model', 'target']).agg(
    median_rmse=('rmse', 'median'),
    mean_rmse=('rmse', 'mean'),
    q25=('rmse', lambda s: s.quantile(0.25)),
    q75=('rmse', lambda s: s.quantile(0.75)),
    n_studies=('study', 'nunique'),
).reset_index()

table4.to_csv(RESULTS / 'table4_loso_summary.csv', index=False)
print('Table 4: LOSO Validation Summary (all models)')
print(table4.round(3))

Table 4: LOSO Validation Summary (all models)
  model  target  median_rmse  mean_rmse     q25      q75  n_studies
0   ERT  P_kbar        3.129      4.322   2.062    5.722         74
1   ERT     T_C       59.618     79.048  41.294   90.502         74
2    GB  P_kbar        2.972      3.922   1.889    4.980         74
3    GB     T_C       60.946     82.252  40.814  101.280         74
4    RF  P_kbar        3.053      4.276   1.972    5.879         74
5    RF     T_C       58.427     79.906  39.933   96.825         74
6   XGB  P_kbar        3.021      4.661   2.159    6.460         74
7   XGB     T_C       67.472     84.189  38.605  104.281         74


## 9.5 Table 5: SHAP Feature Importance (from nb06)

In [6]:
shap_df = pd.read_csv(RESULTS / 'nb06_shap_feature_importance.csv')
shap_df = shap_df.sort_values('importance_P', ascending=False)

table5 = shap_df.head(10)[['feature', 'importance_P', 'importance_T']]
table5.to_csv(RESULTS / 'table5_shap_importance.csv', index=False)
print('Table 5: Top 10 SHAP Feature Importance')
print(table5.round(4))

Table 5: Top 10 SHAP Feature Importance
       feature  importance_P  importance_T
12    liq_SiO2        3.2546       11.7486
16     liq_MgO        1.1867       70.7605
7        Al_VI        0.9040        2.8981
14   liq_Al2O3        0.6347       10.1133
1        Al2O3        0.6209        2.7234
2    FeO_total        0.5852        2.9422
19     liq_K2O        0.4629        2.7835
17     liq_CaO        0.3572        5.6029
18    liq_Na2O        0.3149        1.6000
20  liq_Mg_num        0.2302        3.8039


## 9.6 Compile All Figures

In [7]:
figures = [
    'fig01_pt_distribution.png',
    'fig02_pca_biplot.png',
    'fig03_pred_vs_obs_P.png',
    'fig04_pred_vs_obs_T.png',
    'fig05_model_comparison.png',
    'fig06_loso_boxplot.png',
    'fig07_shap_P_beeswarm.png',
    'fig08_shap_T_beeswarm.png',
    'fig09_shap_P_dependence.png',
    'fig09_shap_T_dependence.png',
    'fig11_bias_correction_residuals.png',
    'fig12_natural_samples_geotherm.png',
    'fig_eda_correlation.png',
    'fig_eda_distributions.png',
]

existing_figs = [f for f in figures if (FIGURES / f).exists()]
print(f'Found {len(existing_figs)}/{len(figures)} figures:')
for f in existing_figs:
    print(f'  {f}')

missing = [f for f in figures if f not in existing_figs]
if missing:
    print(f'\nMissing {len(missing)} figures:')
    for f in missing:
        print(f'  {f}')

key_results = {
    'best_T_rmse': results_df[results_df['target'] == 'T_C']['rmse_test'].min(),
    'best_P_rmse': results_df[results_df['target'] == 'P_kbar']['rmse_test'].min(),
}
pd.DataFrame([key_results]).to_csv(RESULTS / 'manuscript_key_results.csv', index=False)
print(f'\nKey results: best T RMSE = {key_results["best_T_rmse"]:.2f} C, best P RMSE = {key_results["best_P_rmse"]:.3f} kbar')
print('Manuscript compilation complete.')

Found 14/14 figures:
  fig01_pt_distribution.png
  fig02_pca_biplot.png
  fig03_pred_vs_obs_P.png
  fig04_pred_vs_obs_T.png
  fig05_model_comparison.png
  fig06_loso_boxplot.png
  fig07_shap_P_beeswarm.png
  fig08_shap_T_beeswarm.png
  fig09_shap_P_dependence.png
  fig09_shap_T_dependence.png
  fig11_bias_correction_residuals.png
  fig12_natural_samples_geotherm.png
  fig_eda_correlation.png
  fig_eda_distributions.png

Key results: best T RMSE = 123.51 C, best P RMSE = 5.566 kbar
Manuscript compilation complete.
